## Day_16_Pandas_Fundamentals

Part A — answer 15 questions with Pandas (the deliverable)

TASK 01 The 15 Titanic questions

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("titanic.csv")

# Always inspect the data first
print("HEAD:")
print(df.head())

print("\nINFO:")
print(df.info())

# 1. How many passengers are in the dataset?
total_passengers = len(df)
print("\n1. Total passengers : ", total_passengers)

# 2. What fraction of passengers survived overall?
survival_rate = df["survived"].mean()
print("\n2. Overall survived rate : ", survival_rate) 

# 3. What was the survival rate for women versus men?
survived_by_sex = df.groupby("sex")["survived"].mean()
print("\n3. Survival rate by sex :")
print(survived_by_sex)

# 4. What was the survival rate for each passenger class?
survival_by_class = df.groupby("pclass")["survived"].mean()
print("\n4. Survival rate by passenger class :")
print(survival_by_class)

# 5. What was the average age of passengers?
average_age = df["age"].mean()
print("\n5. Average age:", average_age)

# 6. Who was the oldest passenger, and who was the youngest?
oldest = df["age"].max()
youngest = df["age"].min()

print("\n6. Oldest passenger age : ", oldest)
print("Youngest passenger age : ",youngest)

# 7. What was the average fare paid in each class?
average_fare = df.groupby("pclass")["fare"].mean()
print("\n7. Average fare by class :")
print(average_fare)

# 8. How many passengers were children (under 18)?
children = (df["age"] < 18).sum()
print("\n8. Number of CHildren : ", children)

# 9. What was the survival rate among children?
children_survival = df.loc[df["age"] < 18, "survived"].mean()
print("\n9. Children Survival rate : ", children_survival)

# 10. How many passengers embarked from each port?
embarked_counts = df["embarked"].value_counts()
print("\n10. Passengers by embarkation port : ")
print(embarked_counts)

# 11. How many passengers have a missing age?
missing_age = df["age"].isna().sum()
print("\n11. Missing Age : ", missing_age)

# 12. What was the survival rate for women in 1st class?
women_first_class = df[
    (df["sex"] == "female") & (df["pclass"] == 1)
]["survived"].mean()

print("\n12. 1st-class women survival rate : ", women_first_class)

# 13. What was the survival rate for men in 3rd class?
men_third_class = df[
    (df["sex"] == "male") & (df["pclass"] == 3)    
]["survived"].mean()

print("\n13. 3rd-class men survival rate : ", men_third_class)


# 14. What was the average family size on board?
# Family size here = siblings/spouses + parents/children
family_size = df["sibsp"] + df["parch"]
average_family_size = family_size.mean()

print("\n14. Average family size : ", average_family_size)

# 15. Did passengers travelling alone survive at a different rate than those with family?
survival_alone = df.groupby("alone")["survived"].mean()

print("\n15. Survival rate : alone vs family")
print(survival_alone)

HEAD:
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  

INFO:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-nu

### Short Reflection – Pandas and the Titanic Questions

**1. Filter vs. `groupby()`:** For one specific group, filtering is simple and direct; however, `groupby()` is more efficient and cleaner when we want to compare the same result across several groups. For example, `df.groupby('sex')['survived'].mean()` gives the survival rate for both women and men at once.

**2. Definition of “alone”:** A passenger can be defined as travelling alone when `sibsp + parch == 0`, meaning they had no sibling/spouse and no parent/child recorded on board. We can create this with `df['alone'] = (df['sibsp'] + df['parch'] == 0)`.

**3. Most surprising finding:** The most surprising result is that survival differed greatly by **sex and passenger class**, with women and higher-class passengers generally having higher survival rates. This shows that survival on the Titanic was associated with social and demographic factors, not simply being a passenger.

**Overall:** The 15 questions show how Pandas can combine filtering, grouping, aggregation, and new calculated columns to turn raw passenger data into meaningful information.

Part B — handle missing age three ways (+ reflection)


TASK 02 Three strategies for missing age

In [9]:
import pandas as pd

# Load Titanic dataset
df = pd.read_csv("titanic.csv")

# Q1. How many Age values are missing and what fraction of data is missing?
missing = df["age"].isna().sum()
missing_pct = df["age"].isna().mean() * 100

print(f"Q1. Missing Age values : {missing}")
print(f"    Missing fraction   : {missing_pct:.2f}%\n")


# Strategy 1 — Drop rows with missing Age
df_drop = df.dropna(subset=["age"]).copy()

print("Q2. Strategy 1 - Drop missing Age")
print(f"    Rows remaining : {len(df_drop)}")
print(f"    Mean Age       : {df_drop["age"].mean():.2f}")
print(f"    Std Dev        : {df_drop["age"].std():.2f}\n")


# Strategy 2 — Fill missing Age with overall mean
df_mean = df.copy()
mean_age = df["age"].mean()
df_mean["age"] = df_mean["age"].fillna(mean_age)

print("Q3. Strategy 2 - Mean fill")
print(f"    Mean Age       : {df_mean["age"].mean():.2f}")
print(f"    Std Dev        : {df_mean["age"].std():.2f}\n")
print("    Why does SD decrease? Missing values are replaced by the mean,")
print("    which adds values exactly at the center and reduces spread.\n")


# Strategy 3 — Fill Age using median of same class and sex
df_group = df.copy()
df_group["age"] = df_group.groupby(
    ["pclass", "sex"]
)["age"].transform(lambda x: x.fillna(x.median()))

print("Q4. Strategy 3 - Group fill")
print(f"    Mean Age       : {df_group["age"].mean():.2f}")
print(f"    Std Dev        : {df_group["age"].std():.2f}\n")
print("    Why is this smarter? Age differs by passenger class and sex,")
print("    so group medians preserve these differences better than one overall mean.\n")


# Q5. Compare all three strategies side by side
results = pd.DataFrame({
    "Strategy" : ["Drop", "Mean Fill", "Group Fill"],
    "Row" : [len(df_drop), len(df_mean), len(df_group)],
    "Mean Age" : [
        df_drop["age"].mean(),
        df_mean["age"].mean(),
        df_group["age"].mean()
    ],
    "Std Dev" : [
        df_drop["age"].std(),
        df_mean["age"].std(),
        df_group["age"].std()

    ]
})

print("Q5. Strategy comparison")
print(results.round(2).to_string(index=False))



# Q6. Re-run Question 3 — Survival rate for women vs men
def survival_by_sex(data) :
    return data.groupby("sex")["survived"].mean() * 100

print("\nQ6. Survival rate for women vs men:")

for name, data in [
    ("Drop", df_drop),
    ("Mean Fill", df_mean),
    ("Group Fill", df_group)
]:
    rates = survival_by_sex(data)
    print(f"\n{name}:")
    print(f"    Female: {rates.get('female', 0):.2f}%")
    print(f"    Male:   {rates.get('male', 0):.2f}%")



Q1. Missing Age values : 177
    Missing fraction   : 19.87%

Q2. Strategy 1 - Drop missing Age
    Rows remaining : 714
    Mean Age       : 29.70
    Std Dev        : 14.53

Q3. Strategy 2 - Mean fill
    Mean Age       : 29.70
    Std Dev        : 13.00

    Why does SD decrease? Missing values are replaced by the mean,
    which adds values exactly at the center and reduces spread.

Q4. Strategy 3 - Group fill
    Mean Age       : 29.11
    Std Dev        : 13.30

    Why is this smarter? Age differs by passenger class and sex,
    so group medians preserve these differences better than one overall mean.

Q5. Strategy comparison
  Strategy  Row  Mean Age  Std Dev
      Drop  714     29.70    14.53
 Mean Fill  891     29.70    13.00
Group Fill  891     29.11    13.30

Q6. Survival rate for women vs men:

Drop:
    Female: 75.48%
    Male:   20.53%

Mean Fill:
    Female: 74.20%
    Male:   18.89%

Group Fill:
    Female: 74.20%
    Male:   18.89%


# Q7. Reflection

-> The survival rates change only slightly after handling missing Age.

-> This shows that the female-vs-male survival comparison is fairly

-> stable and is not strongly affected by the Age missing-data strategy.

### TASK_2_Reflection – Missing Age Strategies

**Why does filling with a single value shrink the standard deviation?**  
When all missing ages are replaced by the same mean value, new observations are placed exactly at the center of the distribution, so the overall spread becomes smaller and the standard deviation decreases.

**When is dropping rows wrong or fine?**  
Dropping rows can be a poor choice when many values are missing because it wastes useful data and may create bias if the missing values are not random. It can be reasonable when only a small number of rows are missing and removing them is unlikely to affect the analysis.

**Can imputation change conclusions? Is that dangerous?**  
Yes, imputation can change statistical results because the replaced values are estimates rather than original observations. This can be dangerous if the method is inappropriate or not reported, because it may give a misleading impression of the data and affect later conclusions.

**Overall:** Comparing the three strategies shows that missing-data handling is an important statistical decision, and the chosen method should always be clearly reported and justified.

# TASK 03 Reflection: which strategy, and why?

Since your dataset is the **Titanic dataset with 891 rows and 177 missing `age` values (about 19.9%)**, the reflection can be made specific to your actual data rather than generic.

Here is a polished **4–6 sentence Markdown-cell answer** that directly addresses every requirement:

### TASK 03 Reflection: Which Strategy, and Why?

The three missing-age strategies—**dropping missing rows, mean imputation, and median imputation**—each have different effects on the dataset. Dropping the 177 rows with missing ages would preserve the original age distribution but reduce the dataset from 891 to 714 records, which may discard useful information and potentially introduce bias if the missing ages are not random. Mean imputation would keep all 891 records, but it would reduce the natural spread of ages by replacing every missing value with the same average age of about 29.7 years. For this dataset, I would choose **median imputation**, using an age of **28 years**, because it keeps all observations while being less affected by extreme ages and is more appropriate when the data may not be perfectly symmetric. There is no single “correct” strategy because the best choice depends on the goal; for example, if preserving the original observed age distribution were more important than keeping every record, dropping missing values could be preferable, while a more advanced analysis might use group-based or model-based imputation. Finally, transparency is important for trust: imputing around **20% of the dataset** without reporting it could make later results appear more certain or representative than they really are, so the chosen method and its impact should always be clearly documented.

**Why this is strong for your notebook:** it uses your actual numbers (**177 missing ages, 19.9%, mean ≈ 29.7, median = 28**) and explicitly discusses **data loss, spread, possible bias, goal-dependence, and trust/transparency**.


### Reflection: Missing Data and Trust

Imputing missing data is both a **statistical and an ethical choice** because the method can affect results, while researchers also have a responsibility to be transparent about decisions that may influence conclusions. For this dataset, I would choose **median imputation** because it keeps all observations and is less affected by extreme ages, although it can reduce the natural variation in the data. There is no single correct method because the best strategy depends on the purpose of the analysis and the amount and pattern of missing data. To make the analysis trustworthy, I would clearly document the number and percentage of missing values, the chosen method, the reason for choosing it, and any assumptions or limitations. I would also report that approximately **20% of ages were imputed**, so others can understand how the missing-data decision may affect the results and reproduce the analysis. :contentReference[oaicite:0]{index=0}

# Part C — filter, sort, rank

## TASK_04_Multi-condition filters and ranking



In [13]:
import pandas as pd

df = pd.read_csv("titanic.csv")

# 1. Female passengers in 1st or 2nd class and under 30
# --------------------------------------------------------

female_under_30 = df[
    (df["sex"] == "female") &
    (df["pclass"].isin([1, 2])) &
    (df["age"] < 30)
]

print("1. Female passenger in 1st/2nd class and under 30")
print("female_under_30")
print("Number of passengers : ", len(female_under_30))
print("Survival rate : ", round(female_under_30["survived"].mean() * 100, 2), "%")


# 2. Ten oldest passengers who survived
# ---------------------------------------
oldest_survivors = (
    df[df["survived"] == 1].nlargest(10, "age")
    [["sex", "pclass", "age", "fare"]]
)

print("\n2. Ten oldest passengers who survived : ")
print(oldest_survivors.to_string(index=False))


# 3. Compare survival rates of top and bottom 10% fare-payers
# -------------------------------------------------------------

# Find the 90th and 10th percentile fare values
top_10_threshold = df["fare"].quantile(0.90)
bottom_10_threshold = df["fare"].quantile(0.10)

# Select passengers in the top 10% and bottom 10% by fare
top_10_fare = df[df['fare'] >= top_10_threshold]
bottom_10_fare = df[df['fare'] <= bottom_10_threshold]

print("\n3. Fare ranking and survival : ")
print("Top 10% fare threshold : ", round(top_10_threshold, 2))
print("Top 10% survival rate : ", round(top_10_fare["survived"].mean() * 100, 2), "%")

print("Bottom 10% fare threshold:", round(bottom_10_threshold, 2))
print("Bottom 10% survival rate:", round(bottom_10_fare['survived'].mean() * 100, 2), "%")


# 4. Sort the whole dataset by class ascending
#    and fare descending
# ------------------------------------------------

sorted_df = df.sort_values(
    ["pclass", "fare"],
    ascending=[True, False]
)

print("\n4. Dataset sorted by class (ascending) and fare (descending) : ")
print(sorted_df.to_string(index=False))

# 5. Descriptive statistics and value counts
# --------------------------------------------

print("\n5. Descriptive statistics for fare : ")
print(df["fare"].describe())

print("\nDescriptive statistics for age : ")
print(df["age"].describe())


print("\nPassenger count by class : ")
print(df["pclass"].value_counts())

print("\nPassenger count by Embarkation port : ")
print(df["embarked"].value_counts())


1. Female passenger in 1st/2nd class and under 30
female_under_30
Number of passengers :  71
Survival rate :  92.96 %

2. Ten oldest passengers who survived : 
   sex  pclass  age     fare
  male       1 80.0  30.0000
female       1 63.0  77.9583
female       3 63.0   9.5875
  male       2 62.0  10.5000
female       1 62.0  80.0000
female       1 60.0  75.2500
  male       1 60.0  79.2000
female       1 58.0  26.5500
female       1 58.0 146.5208
female       1 58.0 153.4625

3. Fare ranking and survival : 
Top 10% fare threshold :  77.96
Top 10% survival rate :  76.67 %
Bottom 10% fare threshold: 7.55
Bottom 10% survival rate: 14.13 %

4. Dataset sorted by class (ascending) and fare (descending) : 
 survived  pclass    sex   age  sibsp  parch     fare embarked  class   who  adult_male deck embark_town alive  alone
        1       1 female 35.00      0      0 512.3292        C  First woman       False  NaN   Cherbourg   yes   True
        1       1   male 36.00      0      1 512.3292   

# Interesting observations from the results:
- Fare has a large difference between its minimum and maximum values,
  showing that some passengers paid much higher fares than others.
- Age has missing values, since only 714 ages are available out of 891 passengers.
- 3rd class has the largest number of passengers.
- Southampton (S) is the most common embarkation port.

Key results from your dataset: 71 female passengers met the first condition, with a 92.96% survival rate. The top 10% fare-payers had a 76.67% survival rate, compared with 14.13% for the bottom 10%. The dataset contains 491 third-class passengers, and Southampton (S) is the most common embarkation port with 644 passengers.

### TASK 04 – Short Reflection

- **Why use `.isin([1,2])`?** It clearly means “select values that belong to this list,” making the condition shorter and easier to read than writing two separate OR (`|`) conditions. :contentReference[oaicite:0]{index=0}
- **Does higher fare relate to survival?** Yes, in this Titanic dataset, passengers paying higher fares generally had higher survival rates. However, fare may also represent **passenger class and socioeconomic position**, so the relationship should not be interpreted as fare itself causing survival. :contentReference[oaicite:1]{index=1}
- **Sorting vs. ranking:** Sorting rearranges the rows according to a value, while ranking assigns each passenger a numerical position based on that value. For example, sorting by fare puts passengers from lowest to highest fare, while ranking gives each passenger a rank such as 1st, 2nd, or 3rd. :contentReference[oaicite:2]{index=2}
- **Result:** The multi-condition filter, 10 oldest survivors, and top-10% vs. bottom-10% fare survival comparison were computed in the code above.